# PSO Benchmark Analysis

In [1]:
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pso.io.paths import RESULTS_ROOT

# Pick the most recent benchmark run automatically
candidates = sorted(RESULTS_ROOT.glob("benchmarks_*"), reverse=True)
if not candidates:
    raise FileNotFoundError(f"No benchmark results found in {RESULTS_ROOT}")
RESULTS_DIR = candidates[0]
print(f"Using: {RESULTS_DIR}")

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
def parse_curve(val):
    """Parse best_fitness_by_iter stored as string in CSV back to a Python list."""
    if pd.isna(val) or val == "None":
        return None
    try:
        return ast.literal_eval(val)
    except Exception:
        return None

df = pd.read_csv(RESULTS_DIR / "benchmark_results.csv")
df["best_fitness_by_iter"] = df["best_fitness_by_iter"].apply(parse_curve)

print(f"Rows loaded   : {len(df)}")
print(f"Objectives    : {sorted(df['objective'].dropna().unique())}")
print(f"Dimensions    : {sorted(df['dim'].dropna().unique())}")
print(f"Modes         : {sorted(df['mode'].dropna().unique())}")
print(f"Seeds         : {sorted(df['seed'].dropna().unique())}")
df.head(3)

In [ ]:
def parse_curve(val):
    """Parse best_fitness_by_iter stored as string in CSV back to a Python list."""
    if pd.isna(val) or val == "None":
        return None
    try:
        return ast.literal_eval(val)
    except Exception:
        return None

df = pd.read_csv(RESULTS_DIR / "benchmark_results.csv")
df["best_fitness_by_iter"] = df["best_fitness_by_iter"].apply(parse_curve)

print(f"Rows loaded   : {len(df)}")
print(f"Objectives    : {sorted(df['objective'].dropna().unique())}")
print(f"Dimensions    : {sorted(df['dim'].dropna().unique())}")
print(f"Modes         : {sorted(df['mode'].dropna().unique())}")
print(f"Seeds         : {sorted(df['seed'].dropna().unique())}")
df.head(3)

In [ ]:
summary = (
    df.groupby(["objective", "dim", "mode"])["gap_to_optimum"]
    .agg(mean="mean", std="std")
    .reset_index()
)

# Format mean ± std as a readable string for display
summary["mean ± std"] = summary.apply(
    lambda r: f"{r['mean']:.3e} ± {r['std']:.3e}", axis=1
)

pivot = summary.pivot_table(
    index=["objective", "dim"],
    columns="mode",
    values="mean ± std",
    aggfunc="first"
)
print("Gap to optimum — mean ± std across seeds")
pivot

In [ ]:
def pad_curves(curves: list[list[float]]) -> np.ndarray:
    """Pad convergence curves to equal length by repeating the last value."""
    max_len = max(len(c) for c in curves)
    return np.array([c + c[-1:] * (max_len - len(c)) for c in curves])


objectives = sorted(df["objective"].dropna().unique())
dims       = sorted(df["dim"].dropna().unique())
# Exclude pyswarm: it does not expose iteration-level data
modes      = [m for m in sorted(df["mode"].dropna().unique()) if m != "pyswarm"]

for dim in dims:
    sub = df[(df["dim"] == dim) & (df["best_fitness_by_iter"].notna())]
    fig, axes = plt.subplots(1, len(objectives), figsize=(4 * len(objectives), 4))
    fig.suptitle(f"Convergence curves — dim={dim}", fontsize=13, fontweight="bold")

    for ax, obj in zip(axes, objectives):
        ax.set_title(obj)
        ax.set_xlabel("Iteration")
        ax.set_ylabel("Best fitness (log)")
        ax.set_yscale("log")

        for mode in modes:
            curves = [
                c for c in
                sub[(sub["objective"] == obj) & (sub["mode"] == mode)]["best_fitness_by_iter"]
                if c is not None
            ]
            if not curves:
                continue
            mat  = pad_curves(curves)
            mean = mat.mean(axis=0)
            std  = mat.std(axis=0)
            x    = np.arange(1, len(mean) + 1)
            line, = ax.plot(x, mean, label=mode)
            # Shaded band = ±1 std across seeds; clip at 1e-16 so log scale doesn't break
            ax.fill_between(x,
                            np.maximum(mean - std, 1e-16),
                            mean + std,
                            alpha=0.15, color=line.get_color())
        ax.legend(fontsize=8)

    plt.tight_layout()
    out = RESULTS_DIR / f"convergence_dim{dim}.png"
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")

In [ ]:
all_modes = sorted(df["mode"].dropna().unique())
colors    = plt.cm.tab10.colors

for obj in objectives:
    fig, axes = plt.subplots(1, len(dims), figsize=(4 * len(dims), 4))
    fig.suptitle(f"Gap to optimum — {obj}", fontsize=13, fontweight="bold")

    for ax, dim in zip(axes, dims):
        ax.set_title(f"dim={dim}")
        ax.set_ylabel("Gap to optimum (log)")
        ax.set_yscale("log")

        data, labels = [], []
        for mode in all_modes:
            vals = df[
                (df["objective"] == obj) &
                (df["dim"] == dim) &
                (df["mode"] == mode)
            ]["gap_to_optimum"].dropna().values
            if len(vals) > 0:
                # Replace exact zeros so log scale works
                vals = np.where(vals == 0, 1e-16, vals)
                data.append(vals)
                labels.append(mode)

        if data:
            bp = ax.boxplot(data, labels=labels, patch_artist=True, notch=False)
            for patch, color in zip(bp["boxes"], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
            ax.tick_params(axis="x", rotation=25)

    plt.tight_layout()
    out = RESULTS_DIR / f"boxplot_{obj}.png"
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")

In [ ]:
# Only custom PSO rows have comparable timing semantics
custom = df[df["mode"] != "pyswarm"].copy()

mean_times = (
    custom.groupby(["objective", "dim", "mode"])["total_time"]
    .mean()
    .reset_index()
    .rename(columns={"total_time": "mean_time"})
)

pivot_times = mean_times.pivot_table(
    index=["objective", "dim"], columns="mode", values="mean_time"
)

if "sequential" in pivot_times.columns:
    # Speedup = time_sequential / time_X  →  >1 means faster than sequential
    speedup_df = pivot_times.div(pivot_times["sequential"], axis=0)
    speedup_df.columns = [f"speedup_{c}" for c in speedup_df.columns]
    speedup_df = speedup_df.reset_index()
    speedup_df.to_csv(RESULTS_DIR / "speedup_table.csv", index=False)

    print("Speedup vs sequential  (>1 = faster, <1 = slower)")
    display(speedup_df.round(2))

    # Bar chart — one subplot per objective
    speedup_cols = [c for c in speedup_df.columns
                    if c.startswith("speedup_") and "sequential" not in c]

    fig, axes = plt.subplots(1, len(objectives),
                             figsize=(4 * len(objectives), 4), sharey=False)
    fig.suptitle("Speedup vs sequential baseline", fontsize=13, fontweight="bold")

    for ax, obj in zip(axes, objectives):
        ax.set_title(obj)
        ax.set_ylabel("Speedup")
        ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="sequential (1×)")

        sub  = speedup_df[speedup_df["objective"] == obj]
        x    = np.arange(len(sub))
        w    = 0.8 / max(len(speedup_cols), 1)

        for k, col in enumerate(speedup_cols):
            mode_name = col.replace("speedup_", "")
            offset    = (k - len(speedup_cols) / 2 + 0.5) * w
            ax.bar(x + offset, sub[col].values, width=w,
                   label=mode_name, color=colors[k], alpha=0.8)

        ax.set_xticks(x)
        ax.set_xticklabels([f"d={d}" for d in sub["dim"].values])
        ax.legend(fontsize=8)

    plt.tight_layout()
    out = RESULTS_DIR / "speedup_chart.png"
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")
else:
    print("No 'sequential' mode found — cannot compute speedup.")